# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/latent-play/blob/main/notebooks/colab_training_and_demo.ipynb)

This notebook trains and evaluates **`latent-play`** on Google Colab Pro (A100 GPU):
1. **Latent Action Model (LAM)**: Unsupervised discrete action inference ($K=8$) from raw gameplay video (`label_fraction=0.0`)
2. **Action-Conditioned JEPA Dynamics Model**: Predictive representations (`STBlock` spatial-bidirectional + temporal-causal) with EMA target network and **SIGReg** (Cramér-Wold + Epps-Pulley) anti-collapse regularization
3. **Visual Decoder**: Stop-gradient pixel decoder for imaginary rollout visualization
4. **Label-Fraction Sweep & Action Alignment**: Mapping latent codes $k \in \{0..K-1\}$ to environment actions using $0.1\%$ to $100\%$ action labels
5. **Closed-Loop CEM Planning**: Receding-horizon Model Predictive Control directly in representation space

## 1. Setup & Hardware Verification

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Install package and dependencies in Colab
!pip install -q -e ".[crafter,dev]" matplotlib

## 2. Collect Crafter Dataset (300k steps)

In [ ]:
!python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps 300000 --workers 4
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif

## 3. Train Latent Action Model (LAM) on Action-Free Video (`label_fraction=0.0`)

In [ ]:
!python scripts/train_lam.py --config configs/lam_crafter.yaml train.compile=true train.amp=auto

## 4. Train Action-Conditioned JEPA Dynamics Model (with SIGReg Anti-Collapse)

In [ ]:
# 4a. Train latent-play Dynamics conditioned on frozen LAM latent codes (0% action labels)
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.out_dir=checkpoints/dyn_crafter_lam train.amp=auto

# 4b. Train Oracle Upper-Bound Dynamics conditioned on 100% true ground-truth actions
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml train.out_dir=checkpoints/dyn_crafter_oracle train.amp=auto

## 5. Label-Fraction Sweep & Action Alignment (0.1% to 100%)

In [ ]:
!python scripts/sweep_labels.py \
    --data-root data/crafter \
    --lam-ckpt checkpoints/lam_crafter/lam_final.pt \
    --dyn-ckpt checkpoints/dyn_crafter_lam/dyn_final.pt \
    --oracle-ckpt checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --fractions 0.001 0.01 0.05 0.1 1.0 \
    --num-episodes 30

!python scripts/plot_paper_figures.py --metrics-json outputs/sweep_results/sweep_metrics.json --out-dir outputs/figures

## 6. Closed-Loop CEM Planning Visualization (with 1% Action Alignment)

In [ ]:
!python scripts/eval_planning.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --alignment-checkpoint outputs/sweep_results/alignment_f0.01_s0.pt \
    --num-episodes 20

In [ ]:
from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")